<div style="background: linear-gradient(135deg, #1e1b4b, #312e81); color:white; padding:25px; border-radius:10px; 
            text-align:center; font-family:'Segoe UI', sans-serif;">

  <h1 style="margin-bottom:8px;"> WikiArt Image Classification</h1>
  <h3 style="margin-top:0; font-style:italic; font-weight:normal; color:#a5b4fc;">
    Optuna Architecture &amp; Hyperparameter Search
  </h3>

  <hr style="width:60%; border:1px solid #6366f1; margin:15px auto;">

  <p style="margin:5px 0; font-size:15px;">
    <b>Group Project</b> - Deep Learning (2025/2026)
  </p>
  <p style="margin:0; font-size:13px; color:#c7d2fe;">
    Master in Data Science and Advanced Analytics - Nova Information Management School
  </p>
</div>

<br>

<div style="background-color:#1e293b; color:#e0e7ff; padding:15px 20px; border-left:5px solid #6366f1; 
            border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:14px;">
<b>Notebook Description</b><br>
This notebook uses Optuna to jointly search over CNN architecture and training hyperparameters.
Every trial is logged as a row in a persistent CSV file for later analysis.
The search space (layers, filters, activations, dropout, batch norm, pooling, dense depth, learning rate)
is fully defined in a single cell for easy mutation.
</div>

<br>

## 1. Libraries & Setup

In [1]:
import os
import sys
import csv
import datetime
import gc

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


import warnings
import optuna
import tensorflow as tf
import keras
from keras import layers

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath('../src'))

from GPUGuard import GPUGuard
from utils import *

config = load_config('../config.yml')

SEED = config['seed']
set_seeds(SEED)

/home/lucas/Desktop/Deep-learning-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Data Loading

In [2]:
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = config['num_classes']

train_dir = config['paths']['train_dir']
val_dir   = config['paths']['val_dir']
test_dir  = config['paths']['test_dir']

train_ds, val_ds, test_ds, class_names = load_datasets(
    train_dir, val_dir, test_dir,
    img_size=IMG_SIZE, batch_size=BATCH_SIZE,
)

def build_standard_augmentation():
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05, fill_mode="nearest"),
        layers.RandomZoom(0.1),
        layers.RandomBrightness(0.1),
        layers.RandomContrast(0.1),
    ], name='data_augmentation')

    return data_augmentation
    
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# Calculate class weights for imbalanced dataset
class_weights = class_weights(train_ds)


Found 9282 files belonging to 23 classes.
Found 1980 files belonging to 23 classes.
Found 2012 files belonging to 23 classes.
Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']


<div id="search-space" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#a5b4fc;">3. Search Space Definition</h2>
  <p style="color:#c7d2fe; margin-top:8px; margin-bottom:0; font-size:13px;">
    <b>All tunable knobs live here.</b> Edit ranges/choices in this single cell to change what Optuna explores.
  </p>
</div>

In [3]:
SEARCH_SPACE = {
    "n_trials"        : 40,
    "study_name"      : "cnn_search_vFINAL",

    "epochs_per_trial": 35,
    "lr_min"          : 5e-4,
    "lr_max"          : 5e-3,
    "lr_log"          : True,

    "weight_decay_min": 1e-4,
    "weight_decay_max": 1e-2,

    "num_conv_blocks_min": 2,
    "num_conv_blocks_max": 3,

    # Each entry is a valid (non-decreasing) filter progression
    # Encoding the constraint INTO the choices eliminates the clipping bias
    "filters_schedules_2block": [
        [32, 32],
        [32, 64],
        [32, 128],
        [64, 64],
        [64, 128],
    ],
    "filters_schedules_3block": [
        [32, 32,  64],
        [32, 64,  64],
        [32, 64, 128],
        [64, 64, 128],
        [64, 128,128],
    ],

    "conv_activations": ["relu", "elu"],
    "pooling_choices" : ["max", "avg"],
    "use_batch_norm"  : [True, False],
    "conv_dropout_min": 0.0,
    "conv_dropout_max": 0.3,

    "num_dense_layers_min": 1,
    "num_dense_layers_max": 2,

    # Each entry is a valid (non-increasing) unit progression
    "dense_schedules_1layer": [
        [128],
        [256],
    ],
    "dense_schedules_2layer": [
        [256, 256],
        [256, 128],
        [128, 128],
    ],

    "dense_activations": ["relu", "elu"],
    "dense_dropout_min": 0.2,
    "dense_dropout_max": 0.5,

    "csv_path"        : config.get('models', {}).get('optuna', {}).get(
                            'csv', '../models/optuna/optuna_trials.csv'),
    "storage_path"    : "../models/optuna/optuna_study_v6.db",
    "best_model_path" : config.get('models', {}).get('optuna', {}).get(
                            'checkpoint', '../models/optuna/optuna_best/model.keras'),
}

## 4. Model Builder & CSV Logger

In [4]:
def build_model(trial, sp: dict):
    """
    Constraints are encoded in the schedules, not enforced by clipping.
    Optuna sees the schedule index as a single categorical — clean prior.
    """
    # ── Sample architecture ──────────────────────────────────
    n_conv_blocks = trial.suggest_int(
        "n_conv_blocks", sp["num_conv_blocks_min"], sp["num_conv_blocks_max"]
    )

    # Sample one valid schedule directly — no clipping needed
    if n_conv_blocks == 2:
        schedule_key = "filters_schedules_2block"
    else:
        schedule_key = "filters_schedules_3block"

    filters_idx = trial.suggest_int(
        f"filters_schedule_idx_{n_conv_blocks}b",
        0, len(sp[schedule_key]) - 1
    )
    filters_per_block = sp[schedule_key][filters_idx]

    conv_act     = trial.suggest_categorical("conv_act",     sp["conv_activations"])
    pooling_type = trial.suggest_categorical("pooling_type", sp["pooling_choices"])
    use_bn       = trial.suggest_categorical("use_bn",       sp["use_batch_norm"])
    conv_dropout = trial.suggest_float("conv_dropout",
                                       sp["conv_dropout_min"],
                                       sp["conv_dropout_max"])

    n_dense_layers = trial.suggest_int(
        "n_dense_layers", sp["num_dense_layers_min"], sp["num_dense_layers_max"]
    )

    if n_dense_layers == 1:
        dense_key = "dense_schedules_1layer"
    else:
        dense_key = "dense_schedules_2layer"

    dense_idx = trial.suggest_int(
        f"dense_schedule_idx_{n_dense_layers}l",
        0, len(sp[dense_key]) - 1
    )
    dense_units_per_layer = sp[dense_key][dense_idx]

    dense_act    = trial.suggest_categorical("dense_act",    sp["dense_activations"])
    dense_dropout = trial.suggest_float("dense_dropout",
                                        sp["dense_dropout_min"],
                                        sp["dense_dropout_max"])
    lr = trial.suggest_float("lr", sp["lr_min"], sp["lr_max"], log=sp["lr_log"])

    # ── Build graph ──────────────────────────────────────────
    data_augmentation = build_standard_augmentation()

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    for f in filters_per_block:
        x = layers.Conv2D(f, 3, padding="same", activation=conv_act)(x)
        if use_bn:
            x = layers.BatchNormalization()(x)
        if pooling_type == "max":
            x = layers.MaxPooling2D(pool_size=(2, 2))(x)
        else:
            x = layers.AveragePooling2D(pool_size=(2, 2))(x)
        if conv_dropout > 0.0:
            x = layers.SpatialDropout2D(conv_dropout)(x)

    x = layers.GlobalAveragePooling2D()(x)

    for units in dense_units_per_layer:
        x = layers.Dense(units, activation=dense_act)(x)
        if dense_dropout > 0.0:
            x = layers.Dropout(dense_dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name=f"cnn_trial_{trial.number}")

    weight_decay = trial.suggest_float("weight_decay", sp["weight_decay_min"], sp["weight_decay_max"], log=True)

    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.F1Score(average="macro", name="f1_macro"),
        ],
    )

    arch = {
        "n_conv_blocks"    : n_conv_blocks,
        "filters_per_block": filters_per_block,
        "conv_act"         : conv_act,
        "pooling_type"     : pooling_type,
        "use_bn"           : use_bn,
        "conv_dropout"     : round(conv_dropout, 4),
        "n_dense_layers"   : n_dense_layers,
        "dense_units"      : dense_units_per_layer,
        "dense_act"        : dense_act,
        "dense_dropout"    : round(dense_dropout, 4),
        "lr"               : round(lr, 7),
        "weight_decay"     : round(weight_decay, 8),
    }
    return model, arch

CSV_COLUMNS = [
    "timestamp", "trial_number", "state",
    # architecture
    "n_conv_blocks", "filters_per_block", "conv_act", "pooling_type",
    "use_bn", "conv_dropout",
    "n_dense_layers", "dense_units", "dense_act", "dense_dropout",
    # training
    "lr", "weight_decay", "epochs_trained",
    # results
    "val_accuracy", "val_f1_macro",
    "best_val_accuracy", "best_val_f1_macro",
    "n_params",
]

def init_csv(path: str):
    """Create CSV with header only if the file does not yet exist."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        with open(path, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_COLUMNS).writeheader()
        print(f"[CSV] Created new log at: {path}")
    else:
        print(f"[CSV] Appending to existing log: {path}")


def append_csv(path: str, row: dict):
    """Append one trial result row to the persistent CSV."""
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
        writer.writerow(row)

## 5. Optuna Objective & Study

In [5]:
def objective(trial):
    """
    Optuna objective: build, train, log, return val_f1_macro.
    One row is written to the CSV regardless of success or pruning.
    """
    sp  = SEARCH_SPACE
    csv_path = sp["csv_path"]

    # Build model & get architecture description
    model, arch = build_model(trial, sp)
    n_params = model.count_params()

    # Callbacks: early stopping + reduce LR on plateau + checkpoints + gpu guard 
    callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_f1_macro",
        mode="max",
        patience=7,          
        min_delta=0.005,      
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_f1_macro",
        mode="max",
        factor=0.5,
        patience=4,          
        min_delta=0.001,     
        min_lr=1e-6,         
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=sp["best_model_path"].replace(
            ".keras", f"_trial_{trial.number}.keras"
        ),
        monitor="val_f1_macro",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),

    GPUGuard(max_usage_ratio=0.95)
    ]

    row = {
        "timestamp"    : datetime.datetime.now().isoformat(timespec="seconds"),
        "trial_number" : trial.number,
        "state"        : "RUNNING",
        **arch,
        "filters_per_block": str(arch["filters_per_block"]),  # store as string list
        "dense_units"      : str(arch["dense_units"]),
        "epochs_trained"   : 0,
        "val_accuracy"     : None,
        "val_f1_macro"     : None,
        "best_val_accuracy": None,
        "best_val_f1_macro": None,
        "n_params"         : n_params,
    }

    try:
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=sp["epochs_per_trial"],
            callbacks=callbacks,
            class_weight=class_weights,
            verbose=0,
        )

        epochs_trained     = len(history.history["val_accuracy"])
        val_accuracy       = history.history["val_accuracy"][-1]
        val_f1_macro       = history.history["val_f1_macro"][-1]
        best_val_accuracy  = max(history.history["val_accuracy"])
        best_val_f1_macro  = max(history.history["val_f1_macro"])

        row.update({
            "state"            : "COMPLETE",
            "epochs_trained"   : epochs_trained,
            "val_accuracy"     : round(float(val_accuracy),      4),
            "val_f1_macro"     : round(float(val_f1_macro),      4),
            "best_val_accuracy": round(float(best_val_accuracy), 4),
            "best_val_f1_macro": round(float(best_val_f1_macro), 4),
        })
        append_csv(csv_path, row)

        # Print a compact one-liner summary per trial
        print(
            f"Trial {trial.number:>3d} | "
            f"val_f1={best_val_f1_macro:.4f} | "
            f"val_acc={best_val_accuracy:.4f} | "
            f"blocks={arch['n_conv_blocks']} "
            f"filters={arch['filters_per_block']} "
            f"conv_act={arch['conv_act']} "
            f"pool={arch['pooling_type']} "
            f"bn={arch['use_bn']} "
            f"dense={arch['dense_units']} "
            f"lr={arch['lr']:.2e}"
        )
        return float(best_val_f1_macro)

    except optuna.exceptions.TrialPruned:
        row["state"] = "PRUNED"
        append_csv(csv_path, row)
        raise

    except Exception as e:
        row["state"] = f"FAILED: {str(e)[:120]}"
        append_csv(csv_path, row)
        raise optuna.exceptions.TrialPruned()  # skip broken trials w no errors

    finally:
        del model
        keras.backend.clear_session()
        gc.collect()
        tf.keras.backend.clear_session()

## 6. Run Study

In [6]:
sp = SEARCH_SPACE

# Initialise CSV (creates header only on first run; appends on subsequent runs)
init_csv(sp["csv_path"])
os.makedirs(os.path.dirname(sp["storage_path"]), exist_ok=True)


sampler = optuna.samplers.TPESampler(seed=SEED)
pruner  = optuna.pruners.MedianPruner(
    n_startup_trials=5,   # don't prune the first N trials
    n_warmup_steps=5,     # don't prune the first N epochs of any trial
)

study = optuna.create_study(
    study_name     = sp["study_name"],
    direction      = "maximize",   
    sampler        = sampler,
    pruner         = pruner,
    storage=f"sqlite:///{sp['storage_path']}",
    load_if_exists=True
)
# the load if exists gets the db file and continues
# usually if params change I delete all the optuna csvs, keras files and db to start from beggining
study.optimize(
    objective,
    n_trials   = sp["n_trials"],
    timeout    = None,             # set a float (seconds) to add a wall-clock limit
    gc_after_trial=True,        
)

[CSV] Created new log at: ../models/optuna/optuna_trials.csv

Epoch 1: val_f1_macro improved from None to 0.05564, saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 1: finished saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 2: val_f1_macro improved from 0.05564 to 0.09199, saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 2: finished saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 3: val_f1_macro improved from 0.09199 to 0.13422, saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 3: finished saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 4: val_f1_macro improved from 0.13422 to 0.14785, saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 4: finished saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 5: val_f1_macro improved from 0.14785 to 0.16784, saving model to ../models/optuna/optuna_best_trial_0.keras

Epoch 5: finished saving model to ../models/optun